# SAM prompts from spot labels

Turns every spot polygon of `data/<GROUP>/labels/spots` into a normalised bounding box and its centre point, and saves them as `keypoints.pkl` and `bboxes.pkl` (`{image name: array}`) in `data/<GROUP>/keypoints_bboxes/` for the SAM-2 notebook.

In [1]:
import os
import pickle
import numpy as np

In [4]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "fish_ids.csv").is_file())
GROUP = "control"
DATA = ROOT / "data" / GROUP

labels_dir = DATA / "labels" / "spots"
out_dir = DATA / "keypoints_bboxes"
out_dir.mkdir(parents=True, exist_ok=True)


def load_paths(directory):
    """
    Load all files from a directory into a dictionary.

    Dictionary format:
        {
            filename_without_extension: full_file_path
        }
    """
    paths = {}

    for file_name in sorted(os.listdir(directory)):
        key = os.path.splitext(file_name)[0]
        paths[key] = os.path.join(directory, file_name)

    return paths


lbl_paths = load_paths(labels_dir)


def load_bbox_details(label_paths):
    """
    Load polygon annotation details from label files.

    Supported label formats per line:
        class_id x1 y1 x2 y2 ...              (ground truth)
        class_id x1 y1 x2 y2 ... confidence   (predictions)

    This function removes:
        - the first value: class ID
        - the last value: confidence score, only if present
          (detected by an odd number of remaining values)

    Returns:
        dict:
            {
                filename: [
                    [x1, y1, x2, y2, ...],
                    [x1, y1, x2, y2, ...]
                ]
            }
    """
    labels = {}

    for key, path in label_paths.items():
        with open(path, "r") as file:
            lines = file.readlines()

        annotations = []

        for line in lines:
            values = line.split()[1:]

            if not values:
                continue

            if len(values) % 2 == 1:
                values = values[:-1]

            annotations.append(values)

        labels[key] = annotations

    return labels


lbls_det = load_bbox_details(lbl_paths)


def polygon_to_bounding_box(polygon):
    """
    Convert polygon coordinates to a bounding box.

    Args:
        polygon (np.ndarray): Array of shape (N, 2), where each row is (x, y).

    Returns:
        tuple:
            (min_x, min_y, max_x, max_y)
    """
    x_values = polygon[:, 0]
    y_values = polygon[:, 1]

    min_x = float(np.min(x_values))
    max_x = float(np.max(x_values))
    min_y = float(np.min(y_values))
    max_y = float(np.max(y_values))

    return min_x, min_y, max_x, max_y


def create_sam_prompts(labels):
    """
    Create one SAM keypoint and one bounding box per polygon annotation.

    For each polygon:
    - Reshape polygon coordinates into (x, y) pairs.
    - Convert the polygon to a bounding box (x_min, y_min, x_max, y_max).
    - Use the center of the bounding box as the SAM keypoint.

    Returns:
        tuple of dicts:
            sam_key_points: {filename: np.ndarray of shape (num_annotations, 2)}
            sam_bboxes:     {filename: np.ndarray of shape (num_annotations, 4)}
    """
    sam_key_points = {}
    sam_bboxes = {}

    for filename, annotations in labels.items():
        keypoints = []
        bboxes = []

        for annotation in annotations:
            vertices = [float(x) for x in annotation]

            polygon = np.array(vertices).reshape(-1, 2)

            x_min, y_min, x_max, y_max = polygon_to_bounding_box(polygon)
            bboxes.append([x_min, y_min, x_max, y_max])

            center_x = ((x_max - x_min) / 2) + x_min
            center_y = ((y_max - y_min) / 2) + y_min

            keypoints.append([center_x, center_y])

        sam_key_points[filename] = np.array(keypoints)
        sam_bboxes[filename] = np.array(bboxes)

    return sam_key_points, sam_bboxes


sam_key_points, sam_bboxes = create_sam_prompts(lbls_det)


with open(out_dir / "keypoints.pkl", "wb") as file:
    pickle.dump(sam_key_points, file)

with open(out_dir / "bboxes.pkl", "wb") as file:
    pickle.dump(sam_bboxes, file)